In [1]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import Environment, BuildContext
import os

# Connect to workspace
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
print("Connected:", ml_client.workspaces.get("ml-learning-workspace").name)

Found the config file in: /config.json


Connected: ml-learning-workspace


In [2]:
# List existing environments
envs = ml_client.environments.list()
for env in envs:
    print(f"{env.name} - version: {env.version}")

AzureML-AI-Studio-Development - version: None
AzureML-ACPT-pytorch-1.13-py38-cuda11.7-gpu - version: None
AzureML-ACPT-pytorch-1.12-py38-cuda11.6-gpu - version: None
AzureML-ACPT-pytorch-1.12-py39-cuda11.6-gpu - version: None
AzureML-ACPT-pytorch-1.11-py38-cuda11.5-gpu - version: None
AzureML-ACPT-pytorch-1.11-py38-cuda11.3-gpu - version: None
AzureML-responsibleai-0.21-ubuntu20.04-py38-cpu - version: None
AzureML-responsibleai-0.20-ubuntu20.04-py38-cpu - version: None
AzureML-tensorflow-2.5-ubuntu20.04-py38-cuda11-gpu - version: None
AzureML-tensorflow-2.6-ubuntu20.04-py38-cuda11-gpu - version: None
AzureML-tensorflow-2.7-ubuntu20.04-py38-cuda11-gpu - version: None
AzureML-sklearn-1.0-ubuntu20.04-py38-cpu - version: None
AzureML-pytorch-1.10-ubuntu18.04-py38-cuda11-gpu - version: None
AzureML-pytorch-1.9-ubuntu18.04-py37-cuda11-gpu - version: None
AzureML-pytorch-1.8-ubuntu18.04-py37-cuda11-gpu - version: None
AzureML-sklearn-0.24-ubuntu18.04-py37-cpu - version: None
AzureML-lightgbm-

In [3]:
import os

# Create environment folder
os.makedirs("environment", exist_ok=True)

# Create conda specification file
conda_spec = """name: credit-risk-env
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - pip
  - pip:
    - scikit-learn==1.3.0
    - pandas==2.0.3
    - numpy==1.24.3
    - lightgbm==4.0.0
    - shap==0.42.1
    - mlflow==2.7.1
    - azureml-mlflow==1.52.0
    - joblib==1.3.2
"""

with open("environment/conda_spec.yml", "w") as f:
    f.write(conda_spec)

print("Conda spec file created")

Conda spec file created


In [4]:
from azure.ai.ml.entities import Environment

# Create and register custom environment
custom_env = Environment(
    name="credit-risk-environment",
    description="Custom environment for credit risk ML project - includes SHAP, LightGBM, MLflow",
    conda_file="environment/conda_spec.yml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest",
    version="1"
)

registered_env = ml_client.environments.create_or_update(custom_env)
print(f"Environment registered: {registered_env.name}, version: {registered_env.version}")

Environment registered: credit-risk-environment, version: 1


In [5]:
from azure.ai.ml import command, Input
from azure.ai.ml.constants import AssetTypes

# Create a simple test script
import os
os.makedirs("env_test", exist_ok=True)

test_script = '''
import sklearn
import lightgbm
import shap
import mlflow
import pandas as pd
import numpy as np

print(f"scikit-learn version: {sklearn.__version__}")
print(f"lightgbm version: {lightgbm.__version__}")
print(f"shap version: {shap.__version__}")
print(f"mlflow version: {mlflow.__version__}")
print("All packages loaded successfully!")
'''

with open("env_test/test_env.py", "w") as f:
    f.write(test_script)

# Submit a job using the custom environment
test_job = command(
    name="test_custom_env",
    display_name="Test Credit Risk Environment",
    code="env_test",
    command="python test_env.py",
    environment="azureml:credit-risk-environment:1",
    compute="ml-compute-md"
)

returned_job = ml_client.jobs.create_or_update(test_job, experiment_name="environment-test")
print(f"Job submitted: {returned_job.name}")
print(f"View in studio: {returned_job.studio_url}")

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading env_test (0.0 MBs): 100%|███

Job submitted: test_custom_env
View in studio: https://ml.azure.com/runs/test_custom_env?wsid=/subscriptions/749d055e-0977-4255-acd4-54c54916bff8/resourcegroups/ml-learning-rg/workspaces/ml-learning-workspace&tid=5fb38fb2-979b-407e-8873-dec137663a6a
